# CoinGecko crypto market ingest

Consumes the keyless CoinGecko producer topic (`crypto-market`) and appends
one row per coin per producer run to Delta. The 3D crypto dashboard reads an
export of this table from GitHub Pages.

In [ ]:
# Databricks notebook source
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import DoubleType, IntegerType, StringType, StructType

# Shared crypto-message contract produced by producer_crypto.py.
schema = StructType() \
    .add("timestamp", StringType()) \
    .add("coin_id", StringType()) \
    .add("symbol", StringType()) \
    .add("name", StringType()) \
    .add("price", DoubleType()) \
    .add("market_cap", DoubleType()) \
    .add("volume_24h", DoubleType()) \
    .add("change_24h", DoubleType()) \
    .add("market_rank", IntegerType()) \
    .add("trending", IntegerType()) \
    .add("trending_rank", IntegerType()) \
    .add("source", StringType())

aiven_password = dbutils.secrets.get(scope="kafka-scope", key="aiven-password")
jaas_config = f'kafkashaded.org.apache.kafka.common.security.scram.ScramLoginModule required username="avnadmin" password="{aiven_password}";'

kafka_df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka-26289383-md-f1a0.l.aivencloud.com:26452")
    .option("subscribe", "crypto-market")
    .option("startingOffsets", "latest")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "SCRAM-SHA-256")
    .option("kafka.sasl.jaas.config", jaas_config)
    .option("kafka.ssl.truststore.location", "/Volumes/workspace/default/energy_realtime/aiven_ca.pem")
    .option("kafka.ssl.truststore.type", "PEM")
    .load()
)

parsed_df = kafka_df.select(
    from_json(col("value").cast("string"), schema).alias("data")
).select("data.*").where(col("coin_id").isNotNull() & (col("price") >= 0))

query = (parsed_df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", "/Volumes/workspace/default/energy_realtime/checkpoints-crypto")
    .toTable("crypto_market_realtime"))

query.awaitTermination()